# NPCI UPI Ecosystem Reliability & Technical Decline Audit

#### Auditing Bank Core-Banking System (CBS) Downtime & Simulating Smart Rerouting

###  Data Acquisition & Preprocessing

In [4]:
import os
import re
import pandas as pd

def process_and_split_npci_files(data_dir):
    remitter_dfs = []
    beneficiary_dfs = []
    
    files = [f for f in os.listdir(data_dir) if f.endswith(('.csv', '.xlsx', '.xls'))]
    print(f"Found {len(files)} files in directory.")
    
    for file in files:
        file_path = os.path.join(data_dir, file)
        file_lower = file.lower()
        
        # 1. Determine Role Type from filename
        if 'remitter' in file_lower:
            role_type = 'Remitter'
            is_fraction = True  # Remitter files use decimal fractions like 0.9322
        elif 'beneficiary' in file_lower:
            role_type = 'Beneficiary'
            is_fraction = False # Beneficiary files use percentages like 99.97%
        else:
            print(f"Skipping unrecognized file (not remitter/beneficiary): {file}")
            continue
            
        # 2. Extract Month & Year from filename (april-2025)
        match = re.search(r'([a-z]{3,9})[_-]?(\d{2,4})', file_lower)
        if match:
            m_str, y_str = match.groups()
            if len(y_str) == 2:
                y_str = '20' + y_str
            dt = pd.to_datetime(f"{m_str} {y_str}", format='%b %Y', errors='coerce')
            if pd.isna(dt):
                dt = pd.to_datetime(f"{m_str} {y_str}", format='%B %Y', errors='coerce')
            if not pd.isna(dt):
                month_year = dt.strftime('%Y-%m')
            else:
                print(f"Could not parse date from filename: {file}")
                continue
        else:
            print(f"Could not find date pattern in filename: {file}")
            continue

        # 3. Read data, skipping the first subtitle row (header=1)
        try:
            if file_path.endswith(('.xlsx', '.xls')):
                df = pd.read_excel(file_path, header=1)
            else:
                df = pd.read_csv(file_path, header=1)
        except Exception as e:
            print(f"Error reading {file}: {e}")
            continue
            
        # 4. Standardize column names
        df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('%', 'pct')
        
        # 5. Identify columns dynamically
        try:
            bank_col = [c for c in df.columns if 'bank' in c or 'remitter' in c or 'beneficiary' in c][0]
            vol_col = [c for c in df.columns if 'total_volume' in c or 'volume' in c][0]
            app_col = [c for c in df.columns if 'approved' in c and 'pct' in c][0]
            bd_col = [c for c in df.columns if 'bd' in c][0]
            td_col = [c for c in df.columns if 'td' in c][0]
        except IndexError:
            print(f"Column mapping failed for file: {file}. Columns found: {list(df.columns)}")
            continue

        # 6. Build clean dataframe with the requested new columns
        clean_df = pd.DataFrame()
        clean_df['month_year'] = [month_year] * len(df)
        clean_df['bank_name'] = df[bank_col].astype(str).str.strip()
        clean_df['role_type'] = role_type
        
        # 7. Numeric conversions
        clean_df['total_volume_mn'] = pd.to_numeric(df[vol_col].astype(str).str.replace(',', ''), errors='coerce')
        
        def clean_pct(val):
            if pd.isna(val): return 0.0
            val_str = str(val).strip().replace('%', '').replace(',', '')
            try:
                num = float(val_str)
            except ValueError:
                return 0.0
            if is_fraction and 0 <= num <= 1.0:
                num = num * 100
            return round(num, 4)

        clean_df['approved_pct'] = df[app_col].apply(clean_pct)
        clean_df['technical_decline_pct'] = df[td_col].apply(clean_pct)
        clean_df['business_decline_pct'] = df[bd_col].apply(clean_pct)
        
        # 8. Drop rows with missing essential info & take top 50 rows
        clean_df = clean_df.dropna(subset=['bank_name', 'total_volume_mn'])
        clean_df = clean_df[~clean_df['bank_name'].str.lower().isin(['nan', '', 'sr.no.'])]
        clean_df = clean_df.head(50)
        
        # 9. Separate into respective lists
        if role_type == 'Remitter':
            remitter_dfs.append(clean_df)
        else:
            beneficiary_dfs.append(clean_df)
            

    # 10. Combine and Save Remitter files (30 files -> 1 file)
    if remitter_dfs:
        remitter_master = pd.concat(remitter_dfs, ignore_index=True)
        remitter_master['approved_volume_mn'] = (remitter_master['total_volume_mn'] * (remitter_master['approved_pct'] / 100)).round(2)
        remitter_master['total_decline_pct'] = (remitter_master['technical_decline_pct'] + remitter_master['business_decline_pct']).round(4)
        
        remitter_file = os.path.join(data_dir, 'remitter_master_30_files.csv')
        remitter_master.to_csv(remitter_file, index=False)
        print(f"Saved Remitter Master: ({len(remitter_master)} rows)")

    # 11. Combine and Save Beneficiary files (30 files -> 1 file)
    if beneficiary_dfs:
        beneficiary_master = pd.concat(beneficiary_dfs, ignore_index=True)
        beneficiary_master['approved_volume_mn'] = (beneficiary_master['total_volume_mn'] * (beneficiary_master['approved_pct'] / 100)).round(2)
        beneficiary_master['total_decline_pct'] = (beneficiary_master['technical_decline_pct'] + beneficiary_master['business_decline_pct']).round(4)
        
        beneficiary_file = os.path.join(data_dir, 'beneficiary_master_30_files.csv')
        beneficiary_master.to_csv(beneficiary_file, index=False)
        print(f"Saved Beneficiary Master: ({len(beneficiary_master)} rows)")

    # 12. Finally, merge both into 1 single master file with the exact 9 columns
    if remitter_dfs and beneficiary_dfs:
        final_master = pd.concat([remitter_master, beneficiary_master], ignore_index=True)
        
        # 13. Order precisely into your columns
        final_columns = [
            'month_year',
            'bank_name',
            'role_type',
            'total_volume_mn',
            'approved_volume_mn',
            'approved_pct',
            'technical_decline_pct',
            'business_decline_pct',
            'total_decline_pct'
        ]
        final_master = final_master[final_columns]

        # 14. My folder path for both input files and output files
        final_filename = os.path.join(data_dir, 'npci_upi_clean_data.csv')
        final_master.to_csv(final_filename, index=False)
        print(f"\nSUCCESS! Final NPCI single clean CSV created: with {len(final_master)} rows and 9 columns.")


# 14. My folder path for both input files and output files
if __name__ == '__main__':
    my_folder_path = r"D:\DATA_ANALYST_PROJECTS_2026\End_to_End_DATA_ANALYST_PROJECTS\NPCI UPI Ecosystem Reliability & Technical Decline Audit\data"
    process_and_split_npci_files(my_folder_path)

Found 60 files in directory.
Saved Remitter Master: (1500 rows)
Saved Beneficiary Master: (1500 rows)

SUCCESS! Final NPCI single clean CSV created: with 3000 rows and 9 columns.
